In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, make_scorer

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from scipy.sparse import hstack


In [ ]:
# Load the processed dataset
df = pd.read_csv("processed.csv")
df.head()

with open("data/stopwords_en.txt", "r") as f:
    custom_stopwords = [line.strip() for line in f if line.strip()]



In [17]:
# Split once for fair comparison
train_text, test_text, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=42, stratify=y
)

# --- Vectorizers with custom stopwords ---
tfidf_title = TfidfVectorizer(lowercase=True, stop_words=custom_stopwords)
tfidf_review = TfidfVectorizer(lowercase=True, stop_words=custom_stopwords)


In [18]:
# --- Task 3.1: Description (Review Text) ---
X_train_review = tfidf_review.fit_transform(train_text["Review Text"].astype(str))
X_test_review = tfidf_review.transform(test_text["Review Text"].astype(str))




/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ain', 'aren', 'couldn', 'didn', 'doesn', 'don', 'hadn', 'hasn', 'haven', 'isn', 'll', 'mon', 'shouldn', 've', 'wasn', 'weren', 'won', 'wouldn'] not in stop_words.
  warnings.warn(


In [19]:
# --- Task 3.2: Title ---
X_train_title = tfidf_title.fit_transform(train_text["Title"].astype(str))
X_test_title = tfidf_title.transform(test_text["Title"].astype(str))



/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ain', 'aren', 'couldn', 'didn', 'doesn', 'don', 'hadn', 'hasn', 'haven', 'isn', 'll', 'mon', 'shouldn', 've', 'wasn', 'weren', 'won', 'wouldn'] not in stop_words.
  warnings.warn(


In [20]:
# --- Task 3.3: Title + Review ---
X_train_combined = hstack([X_train_title, X_train_review])
X_test_combined = hstack([X_test_title, X_test_review])

In [5]:
# Classifier
classifier = LogisticRegression(max_iter=2000, solver='saga')

In [6]:
# 5-Fold Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=36)
accuracy_scorer = make_scorer(accuracy_score)
f1_scorer = make_scorer(f1_score)

In [7]:
# Cross-validation
accuracy = cross_val_score(classifier, X_title, y, cv=skf, scoring=accuracy_scorer)
f1 = cross_val_score(classifier, X_title, y, cv=skf, scoring=f1_scorer)

In [8]:
# Print results
print("Title Only Model")
print("Accuracy:", accuracy.mean())
print("F1 Score:", f1.mean())

Title Only Model
Accuracy: 0.88170055448084
F1 Score: 0.9302401086782414


In [9]:
# sample test
test_review = ["This product is great! I loved it.", "This is the most disappointing purchase I've made.", "The quality is terrible", "Absolutely fantastic!"]

x_test = tfidf_vectorizer.transform(test_review)
test_classifier = LogisticRegression(max_iter=2000, solver="saga")
test_classifier.fit(X_title, y)
test_pred = test_classifier.predict(x_test) 

test_pred

array([1, 0, 0, 1])